# Propagation

This notebook focuses on what happens after power leaves the antenna: free-space loss, Fresnel zone clearance, and how frequency changes the geometry of a path.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Free-Space Path Loss Across Bands

Higher frequency means shorter wavelength, which increases free-space loss for the same path length.

In [ ]:
distances = np.logspace(-1, 2, 300)
freqs_mhz = {
    "HF 7 MHz": 7,
    "VHF 146 MHz": 146,
    "UHF 462 MHz": 462,
    "ISM 915 MHz": 915,
    "Wi-Fi 2400 MHz": 2400,
}

fig, ax = plt.subplots(figsize=(10, 4))
for label, freq in freqs_mhz.items():
    ax.plot(distances, fspl_db(distances, freq), label=label)
ax.set_xscale("log")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("FSPL (dB)")
ax.set_title("Path loss by frequency")
ax.legend()
plt.tight_layout()


## Fresnel Zone Width

Line of sight is not enough by itself. Objects intruding into the Fresnel zone can still damage the path.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

def fresnel_radius(d1_km, d2_km, freq_mhz):
    wavelength = 300 / freq_mhz
    d1 = d1_km * 1000
    d2 = d2_km * 1000
    return np.sqrt(wavelength * d1 * d2 / (d1 + d2))

def update_fresnel(total_distance_km=10.0, freq_mhz=462.0):
    ax.clear()
    x = np.linspace(0, total_distance_km, 400)
    radius = fresnel_radius(total_distance_km / 2, total_distance_km / 2, freq_mhz)
    profile = radius * np.sqrt(np.clip(1 - ((x - total_distance_km / 2) / (total_distance_km / 2)) ** 2, 0, None))
    ax.plot(x, profile, label="1st Fresnel zone")
    ax.plot(x, -profile, color="tab:blue")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Path distance (km)")
    ax.set_ylabel("Radius (m)")
    ax.set_title(f"Mid-path Fresnel radius: {radius:.2f} m")
    ax.legend()
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_fresnel,
    total_distance_km=float_slider(min_value=1, max_value=50, step=1, value=10, description="Distance km"),
    freq_mhz=float_slider(min_value=30, max_value=2400, step=10, value=462, description="Freq MHz"),
)
display(controls)


## Key Takeaway

Propagation is geometry plus wavelength. Distance, frequency, and clearance determine whether a path is clean, marginal, or doomed.